# <b>The Skeleton Run

---

---

# 1. The Setup

## 1. Importing Libraries

In [37]:
import os
import torch
import torch.nn as nn

import pymupdf
import re
import fitz

from transformers import(
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
    DistilBertModel
)

from peft import PeftModel

## 2. Device Configuration

In [2]:
device = "mps"

## 3. Locate the models.

In [21]:
BASE_DIR = "../models/"

### 3.1 Locate the text-classifier

In [39]:
MIRA_DISTILBERT_LORA = f"{BASE_DIR}/mira-distilbert-lora"

### 3.2 Locate the text-classifier heads.

In [44]:
MIRA_CLASSIFIER_HEADS = f"{BASE_DIR}/mira-distilbert-lora"

### 3.3 Locate the Risk Engine.

In [46]:
MIRA_RISK_CLASSIFIER = f"{BASE_DIR}/mira_risk_engine/mira-risk-classifier.pkl"

In [49]:
MIRA_RISK_ENCODER = f"{BASE_DIR}/mira_risk_engine/mira-risk-encoder.pkl"

### 3.3 Locate the GenLLM.

In [25]:
GEN_MODEL_PATH = f"{BASE_DIR}/MIRA2_qwen2.5-3b-lora"

---

## 4. Load <b>Inspection Classifier</b> for PDF's text.

In [11]:
class MIRAClassifier(nn.Module):
    def __init__(self, num_issue, num_category, num_severity, num_recurring):
        super().__init__()

        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')

        hidden_size = self.distilbert.config.hidden_size

        # sub-models
        self.issue_classifier = nn.Linear(hidden_size, num_issue)
        self.category_classifier = nn.Linear(hidden_size, num_category)
        self.severity_classifier = nn.Linear(hidden_size, num_severity)
        self.recurring_classifier = nn.Linear(hidden_size, num_recurring)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids = input_ids,
            attention_mask = attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        issue_logits = self.issue_classifier(pooled_output)
        category_logits = self.category_classifier(pooled_output)
        severity_logits = self.severity_classifier(pooled_output)
        recurring_logits = self.recurring_classifier(pooled_output)

        return {
            'issue': issue_logits,
            'category': category_logits,
            'severity': severity_logits,
            'recurring': recurring_logits
        }

In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [12]:
num_issue = 10
num_category = 7
num_severity = 4
num_recurring = 2

In [38]:
inspection_classifier = MIRAClassifier(
    num_issue = num_issue,
    num_category = num_category,
    num_severity = num_severity,
    num_recurring = num_recurring
)

In [42]:
inspection_classifier.distilbert = PeftModel.from_pretrained(
    inspection_classifier.distilbert,
    MIRA_DISTILBERT_LORA
)

### 4.1 Load Classification Heads

In [43]:
heads = torch.load(
    os.path.join(
        MIRA_CLASSIFIER_HEADS,
        "classification_heads.pth"
    ),
    map_location=device
)

inspection_classifier.issue_classifier.load_state_dict(
    heads["issue_classifier"]
)

inspection_classifier.category_classifier.load_state_dict(
    heads["category_classifier"]
)

inspection_classifier.severity_classifier.load_state_dict(
    heads["severity_classifier"]
)

inspection_classifier.recurring_classifier.load_state_dict(
    heads["recurring_classifier"]
)

inspection_classifier = inspection_classifier.to(device)
inspection_classifier.eval()

MIRAClassifier(
  (distilbert): PeftModelForFeatureExtraction(
    (base_model): LoraModel(
      (model): PeftModelForFeatureExtraction(
        (base_model): LoraModel(
          (model): DistilBertModel(
            (embeddings): Embeddings(
              (word_embeddings): Embedding(30522, 768, padding_idx=0)
              (position_embeddings): Embedding(512, 768)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (transformer): Transformer(
              (layer): ModuleList(
                (0-5): 6 x TransformerBlock(
                  (attention): MultiHeadSelfAttention(
                    (dropout): Dropout(p=0.1, inplace=False)
                    (q_lin): lora.Linear(
                      (base_layer): Linear(in_features=768, out_features=768, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.1, 

### 4.2 Load Label Mappings

In [ ]:
import json

with open(
    os.path.join(
        MIRA_DISTILBERT_LORA,
        'label_mappings.json'
    ),
    'r'
) as f:
    label_mappings = json.load(f)

---

## 5. Load <b>Risk Engine

In [51]:
import joblib

risk_model = joblib.load(MIRA_RISK_CLASSIFIER)
risk_encoder = joblib.load(MIRA_RISK_ENCODER)

---

## 6. Load <b>GenLLM</b>

In [53]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LORA_PATH = GEN_MODEL_PATH

In [54]:
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)

In [55]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype = torch.float16
).to(device)

model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

---

---